In [ ]:
import nibabel as nib
import numpy as np
from pathlib import Path

data_dir = Path("data/Merlin Abdominal CT Dataset_files")
files = [f for f in sorted(data_dir.glob("*.nii.gz"))]

print(files)
# img = nib.load(files[0])
# volume = img.get_fdata()

# print(volume)
# print(volume.shape)
# print(volume.dtype)

# print(volume.min())
# print(volume.max())

# img.header.get_zooms()

In [24]:
FILTERED_FINDING_PHRASES = {
    "renal_cyst":                     ("a renal cyst is present", "no renal cyst"),
    "surgically_absent_gallbladder":  ("the gallbladder is surgically absent", "the gallbladder is present"),
    "atelectasis":                    ("atelectasis is present", "no atelectasis"),
    "pleural_effusion":               ("a pleural effusion is present", "no pleural effusion"),
}

In [ ]:
from merlin.data import DataLoader
from merlin import Merlin
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# datalist = [{"image": f, "study_id": f.stem.removesuffix(".nii")} for f in files]
datalist = [{"image": files[0], "study_id": "AC421363e"}, {"image": files[1], "study_id": "AC421363f"}]

dataloader = DataLoader(
    datalist=datalist,
    cache_dir=None,
    batchsize=1,
    shuffle=True,
    num_workers=0,
)

# model_img = Merlin(ImageEmbedding=True)
# model_img.eval().cuda()

model = Merlin()
model.eval().cuda()

flat_phrases, phrase_keys = [], []

for finding, (present, absence) in FILTERED_FINDING_PHRASES.items():
    flat_phrases += [present, absence]
    phrase_keys += [(finding, "present"), (finding, "absent")]

img_embeds = {}
dummy_tensor = None
with torch.no_grad():
    for batch in dataloader:
        sid = batch["study_id"][0]
        if dummy_tensor is None:
            dummy_tensor = batch["image"].cuda()
        img_out, _, _ = model(batch["image"].cuda(), flat_phrases)
        img_embeds[sid] = img_out[0]

with torch.no_grad():
    _, _, text_embeds = model(dummy_tensor, flat_phrases)

text_lookup = {}
for i, (finding, polarity) in enumerate(phrase_keys):
    text_lookup.setdefault(finding, {})[polarity] = text_embeds[i]



In [ ]:
import torch.nn.functional as F
import json

results = {}

with torch.no_grad():
    for finding in FILTERED_FINDING_PHRASES:
        sid = list(img_embeds.keys())

        y_score = []

        for sid_i in sid:
            sim_present = F.cosine_similarity(img_embeds[sid_i], text_lookup[finding]["present"], dim=0)
            sim_absent = F.cosine_similarity(img_embeds[sid_i], text_lookup[finding]["absent"], dim=0)

            probs = F.softmax(torch.stack([sim_present, sim_absent]), dim=0)
            score = probs[0].item()

            y_score.append(score)

        results[finding] = {
            "study_id": sid,
            "y_score": y_score
        }

        with open("merlin_zero_shot_results.json", "w") as f:
            json.dump(results, f, indent=2)